# SensorThings API — Registering STA entities

This notebook shows how to create the main SensorThings API entities used by istSOS4.

The goal is to build a small but complete example:

1. create a `Network`;
2. create a monitored object, called a `Thing`;
3. assign a `Location` to the `Thing`;
4. define what is measured with an `ObservedProperty`;
5. define the instrument with a `Sensor`;
6. connect everything through a `Datastream`;
7. send one or more `Observations`.

## Main entities used in this notebook

| Entity | Simple meaning | Example in this notebook | Why it is needed |
|---|---|---|---|
| `Network` | A group or project that contains related stations or datastreams | `DDT_workshop` | Useful to organize data by project, campaign, or network |
| `Thing` | The physical or logical object being monitored | A river monitoring station | Represents what the observations belong to |
| `Location` | Where the `Thing` is located | Coordinates of the station | Places the monitored object on the map |
| `ObservedProperty` | What is being measured | Water voltage | Describes the measured variable |
| `Sensor` | The device or method used to measure | `Ecolog 1000` | Describes how the value is produced |
| `Datastream` | The connection between `Thing`, `Sensor`, and `ObservedProperty` | Voltage measurements from the station | Groups observations of the same type |
| `Observation` | A measured value | `3.63` | Stores the actual data value |
| `FeatureOfInterest` | The specific feature observed by an observation | The station location or another point | Describes what the observation refers to |

## Authorization note

This notebook assumes that the authorization notebook has already been completed.

The administrator has full access to the system and can create users, policies, and data.  
In this notebook, the administrator is used only to create the `Network`.

After that, we log in as an `editor`. The `editor` user creates the SensorThings entities and observations. This shows that the editor policy allows managing the system resources required for this workflow.

> If the `Network` extension is disabled in your deployment, skip the `Network` creation step and remove the `Network` field from the `Datastream` payloads.


## Step 1 — Preliminary setup

First, we import the required Python libraries and define the base URL of the istSOS4 API.

When this notebook runs inside the Jupyter Docker container, the API should be reached through the Docker service name:

```python
IST_SOS_ENDPOINT = "http://api:5000/v4/v1.1"
```

If you run the same code directly from your host machine instead of inside Docker, you may need to use the external port exposed by Docker Compose.


In [1]:
import json

import requests
from IPython.display import Markdown, display

from istsos_utils import (
    REQUEST_TIMEOUT,
    auth_headers,
    display_error_response,
    display_json,
    get_or_create_network,
    login,
    print_created,
)

IST_SOS_ENDPOINT = "http://api:5000/v4/v1.1"

## Step 2 — Login as administrator

We first log in as the administrator.

The administrator has full access to the system. In this notebook, we use the administrator only for the initial `Network` creation. The rest of the entities are created with the `editor` user.


In [2]:
admin_username = input("Enter administrator username: ")
admin_password = input("Enter administrator password: ")

if not admin_username or not admin_password:
    print("Username or password is empty")
else:
    admin_token, login_body = login(
        IST_SOS_ENDPOINT,
        admin_username,
        admin_password,
        timeout=REQUEST_TIMEOUT,
    )

    if admin_token:
        print("Logged in as administrator")

Enter administrator username:  admin
Enter administrator password:  admin


Logged in as administrator


## Step 3 — Create a `Network`

A `Network` is used to group related datastreams, stations, or monitoring resources.

In this workshop we create one network called `DDT_workshop` and store its identifier in `network_id`.

> This step requires the `Network` extension to be enabled in the istSOS4 deployment.


In [3]:
network_name = "DDT_workshop"

network_id = get_or_create_network(
    IST_SOS_ENDPOINT,
    admin_token,
    network_name,
    timeout=REQUEST_TIMEOUT,
)

print(f"Network ID: {network_id}")

Network created successfully (http://localhost:8018/v4/v1.1/Networks(1))
Network ID: 1


## Step 4 — Login as editor

From now on, we use an `editor` user.

The `editor` role is allowed to create and manage the SensorThings entities used in this example.

The username is also used as a prefix for the names of the resources created in the notebook. This avoids name collisions when multiple people run the workshop at the same time.


In [4]:
editor_username = input("Enter editor username: ")
editor_password = input("Enter editor password: ")

if not editor_username or not editor_password:
    print("Username or password is empty")
else:
    editor_token, login_body = login(
        IST_SOS_ENDPOINT,
        editor_username,
        editor_password,
        timeout=REQUEST_TIMEOUT,
    )

    if editor_token:
        prefix = editor_username + "-"
        print("Logged in as editor")
        print("Resource names will be prefixed with: " + prefix)

Enter editor username:  group1_editor
Enter editor password:  qwertz


Logged in as editor
Resource names will be prefixed with: group1_editor-


## Step 5 — Create a `Thing`

A `Thing` represents the object being monitored.

In this example, the `Thing` is a river monitoring station. It is the central object to which the location and datastreams will be connected.

The created identifier is stored in `thing_id`.


In [5]:
thing_body = {
    "name": f"{prefix}FIU_VAL",
    "description": "Water level, water temperature and water electrical conductivity recorder on the Ticino river",
    "properties": {
        "keywords": "water, river, height, temperature, conductivity, ACSOT",
        "description": "River level, water temperature and water electrical conductivity at Fiume Ticino Valle",
    },
}

response = requests.post(
    IST_SOS_ENDPOINT + "/Things",
    json=thing_body,
    headers=auth_headers(editor_token, "Create new thing"),
    timeout=REQUEST_TIMEOUT,
)

if response.status_code == 201:
    thing_id = print_created("Thing", response)
else:
    display_error_response(response)

Thing created successfully (http://localhost:8018/v4/v1.1/Things(3))


## Step 6 — Create a `Location`

A `Location` describes where a `Thing` is placed.

Here we create a point geometry using Swiss coordinates in `EPSG:2056` and link it to the `Thing` created in the previous step.

The created identifier is stored in `location_id`.


In [6]:
location_body = {
    "name": f"{prefix}Fiume Ticino Valle",
    "description": "Location of the river monitoring station",
    "encodingType": "application/vnd.geo+json",
    "location": {
        "type": "Point",
        "coordinates": [
            2717185.973,
            1114552.035,
        ],
        "crs": {
            "type": "name",
            "properties": {
                "name": "EPSG:2056",
            },
        },
    },
    "Things": [
        {"@iot.id": thing_id},
    ],
}

response = requests.post(
    IST_SOS_ENDPOINT + "/Locations",
    json=location_body,
    headers=auth_headers(editor_token, "Create new location"),
    timeout=REQUEST_TIMEOUT,
)

if response.status_code == 201:
    location_id = print_created("Location", response)
else:
    display_error_response(response)

Location created successfully (http://localhost:8018/v4/v1.1/Locations(1))


## Step 7 — Create an `ObservedProperty`

An `ObservedProperty` describes what is being measured.

In this example, we create an observed property for voltage. Later, the datastream will use this property to say that its observations are voltage measurements.

The created identifier is stored in `observed_property_id`.


In [7]:
observed_property_body = {
    "name": f"{prefix}ground:water:voltage",
    "description": "Ground water voltage",
    "properties": {},
    "definition": "{}",
}

response = requests.post(
    IST_SOS_ENDPOINT + "/ObservedProperties",
    json=observed_property_body,
    headers=auth_headers(editor_token, "Create new ObservedProperty"),
    timeout=REQUEST_TIMEOUT,
)

if response.status_code == 201:
    observed_property_id = print_created("ObservedProperty", response)
else:
    display_error_response(response)

ObservedProperty created successfully (http://localhost:8018/v4/v1.1/ObservedProperties(1))


## Step 8 — Create a `Sensor`

A `Sensor` describes the device, instrument, or method used to produce the measurements.

In this example, we create a sensor called `Ecolog 1000`.

The created identifier is stored in `sensor_id`.


In [8]:
sensor_body = {
    "name": f"{prefix}Ecolog 1000",
    "description": "Pressure, temperature and electrical conductivity sensor",
    "properties": {},
    "encodingType": "application/json",
    "metadata": '{"brand": "OTT", "type": "Pressure, temperature, electrical conductivity sensor"}',
}

response = requests.post(
    IST_SOS_ENDPOINT + "/Sensors",
    json=sensor_body,
    headers=auth_headers(editor_token, "Create new Sensor"),
    timeout=REQUEST_TIMEOUT,
)

if response.status_code == 201:
    sensor_id = print_created("Sensor", response)
else:
    display_error_response(response)

Sensor created successfully (http://localhost:8018/v4/v1.1/Sensors(1))


## Step 9 — Create a `Datastream`

A `Datastream` connects the main parts of the SensorThings model:

- the `Thing` that is being monitored;
- the `Sensor` that produces the measurements;
- the `ObservedProperty` that describes what is measured;
- the unit of measurement;
- optionally, the `Network` used to organize the data.

After the datastream is created, observations can be posted to it.

The created identifier is stored in `datastream_id`.


In [9]:
datastream_body = {
    "name": f"{prefix}V_FIU_VAL",
    "description": "Voltage datastream for the Fiume Ticino Valle station",
    "observationType": "http://www.opengis.net/def/observationType/OGC-OM/2.0/OM_Measurement",
    "unitOfMeasurement": {
        "name": "Voltage",
        "symbol": "V",
        "definition": "",
    },
    "properties": {
        "samplingFrequency": "PT60M",
        "acquisitionFrequency": "PT60M",
    },
    "Thing": {"@iot.id": thing_id},
    "Sensor": {"@iot.id": sensor_id},
    "ObservedProperty": {"@iot.id": observed_property_id},
    "Network": {"@iot.id": network_id},
}

response = requests.post(
    IST_SOS_ENDPOINT + "/Datastreams",
    json=datastream_body,
    headers=auth_headers(editor_token, "Create new Datastream"),
    timeout=REQUEST_TIMEOUT,
)

if response.status_code == 201:
    datastream_id = print_created("Datastream", response)
else:
    display_error_response(response)

Datastream created successfully (http://localhost:8018/v4/v1.1/Datastreams(1))


## Step 10 — Create observations

An `Observation` is the actual measured value.
If `phenomenonTime` is not provided, the server assigns the current time automatically.


### Example 1 — Observation with only a datastream

In this first example, we send only the result value and the datastream.

The server can derive the `FeatureOfInterest` from the location of the `Thing` connected to the datastream.


In [10]:
observation_body = {
    "result": 3.63,
    "Datastream": {"@iot.id": datastream_id},
}

response = requests.post(
    IST_SOS_ENDPOINT + "/Observations",
    json=observation_body,
    headers=auth_headers(editor_token, "Create new Observation"),
    timeout=REQUEST_TIMEOUT,
)

if response.status_code == 201:
    observation_id = print_created("Observation", response)
else:
    display_error_response(response)

Observation created successfully (http://localhost:8018/v4/v1.1/Observations(1))


## Step 11 — Create multiple related entities in one request

The SensorThings API also allows creating related entities together in a single request.

In this example, we create a new `Datastream` and include several `Observations` directly inside the same JSON payload.

This approach is useful when importing small groups of related data, because the relationships are defined in the same request.


In [11]:
datastream_with_observations_body = {
    "name": f"{prefix}RSSI_FIU_VAL",
    "description": "RSSI datastream for the Fiume Ticino Valle station",
    "observationType": "http://www.opengis.net/def/observationType/OGC-OM/2.0/OM_Measurement",
    "unitOfMeasurement": {
        "name": "",
        "symbol": "RSSI",
        "definition": "",
    },
    "properties": {
        "samplingFrequency": "PT60M",
        "acquisitionFrequency": "PT60M",
    },
    "ObservedProperty": {"@iot.id": observed_property_id},
    "Sensor": {"@iot.id": sensor_id},
    "Thing": {"@iot.id": thing_id},
    "Network": {"@iot.id": network_id},
    "Observations": [
        {
            "result": 1,
        },
        {
            "result": 2,
        },
        {
            "result": 3,
        },
    ],
}

response = requests.post(
    IST_SOS_ENDPOINT + "/Datastreams",
    json=datastream_with_observations_body,
    headers=auth_headers(editor_token, "Create Datastream and related Observations"),
    timeout=REQUEST_TIMEOUT,
)

if response.status_code == 201:
    second_datastream_id = print_created("Datastream", response)
else:
    display_error_response(response)

Datastream created successfully (http://localhost:8018/v4/v1.1/Datastreams(2))


## Summary

In this notebook we created a complete SensorThings data structure:

| Step | Entity | Stored identifier |
|---|---|---|
| 1 | `Network` | `network_id` |
| 2 | `Thing` | `thing_id` |
| 3 | `Location` | `location_id` |
| 4 | `ObservedProperty` | `observed_property_id` |
| 5 | `Sensor` | `sensor_id` |
| 6 | `Datastream` | `datastream_id` |
| 7 | `Observation` | `observation_id` |
| 8 | Second `Datastream` with nested observations | `second_datastream_id` |

The most important relationship is the `Datastream`: it connects the monitored object, the sensor, the observed property, and the observations.
